# Bronze Layer — Raw Ingestion
Loads every source CSV from the `src` volume into a Delta table under `workspace.bronze`, as-is (schema inferred, no transformations).

Bronze tables are a faithful copy of the source so we can always rebuild silver/gold from here.

Every row gets two audit columns:
- `_ingested_at` — when this load ran
- `_source_file` — which file the row came from

## Ingestion config
One entry per source file. Adding a new source = adding a row here.

In [0]:
CATALOG = "workspace"
SCHEMA = "bronze"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/src"

INGESTION_CONFIG = [
    {"source": "crm", "file": "src_crm/cust_info.csv",     "table": "crm_cust_info"},
    {"source": "crm", "file": "src_crm/prd_info.csv",      "table": "crm_prd_info"},
    {"source": "crm", "file": "src_crm/sales_details.csv", "table": "crm_sales_details"},
    {"source": "erp", "file": "src_erp/CUST_AZ12.csv",     "table": "erp_cust_az12"},
    {"source": "erp", "file": "src_erp/LOC_A101.csv",      "table": "erp_loc_a101"},
    {"source": "erp", "file": "src_erp/PX_CAT_G1V2.csv",   "table": "erp_px_cat_g1v2"},
]

## Ingest files into bronze tables

In [0]:
import pyspark.sql.functions as F

for item in INGESTION_CONFIG:
    path = f"{VOLUME_ROOT}/{item['file']}"
    table = f"{CATALOG}.{SCHEMA}.{item['table']}"
    print(f"Ingesting {item['source']}: {path} -> {table}")

    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(path)
             .withColumn("_ingested_at", F.current_timestamp())
             .withColumn("_source_file", F.col("_metadata.file_name"))
    )

    (
        df.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable(table)
    )

    print(f"  {df.count()} rows written")

## Sanity check

In [0]:
%sql
SHOW TABLES IN workspace.bronze;

In [0]:
%sql
SELECT _source_file, _ingested_at, COUNT(*) AS rows
FROM workspace.bronze.crm_cust_info
GROUP BY ALL;